In [3]:
import kagglehub
import pandas as pd 
import numpy as np
import torch 
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader 
import torch.optim as optim 
import matplotlib.pyplot as plt
import torch.nn as nn 


In [4]:
torch.__version__ 

'2.9.0+cu126'

In [5]:
device = torch.device('cuda')

In [6]:
path = kagglehub.dataset_download("zalando-research/fashionmnist")
print("Path to dataset files ", path)

Using Colab cache for faster access to the 'fashionmnist' dataset.
Path to dataset files  /kaggle/input/fashionmnist


In [7]:
%pip install opendatasets


In [8]:
import opendatasets as od
od.download(
    "https://www.kaggle.com/datasets/zalando-research/fashionmnist"
)

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username:Your Kaggle Key:Dataset URL: https://www.kaggle.com/datasets/zalando-research/fashionmnist


100%|██████████| 68.8M/68.8M [00:00<00:00, 1.71GB/s]

In [ ]:
key = KGAT_855e10921136a5797597d02a8d21b456

In [9]:
!ls /content
!ls /content/fashionmnist


fashionmnist  sample_data
fashion-mnist_test.csv	 t10k-images-idx3-ubyte  train-images-idx3-ubyte
fashion-mnist_train.csv  t10k-labels-idx1-ubyte  train-labels-idx1-ubyte


In [10]:
train_df = pd.read_csv('/content/fashionmnist/fashion-mnist_train.csv')
test_df  = pd.read_csv("/content/fashionmnist/fashion-mnist_test.csv")

print(train_df.shape)
train_df.head()


(60000, 785)


,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [11]:
X = train_df.iloc[:,1:].values
y = train_df.iloc[:,0].values

In [12]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [14]:
# transformation 
from torchvision.transforms import transforms

custom_transform = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]
)

In [30]:
from PIL import Image
import numpy as np 
class CustomDataset(Dataset):
    def __init__(self,features, labels, transforms):
        self.features = features
        self.labels = labels
        self.transforms = transforms

    def __len__(self):
        return len(self.features)

    def __getitem__(self,index):
        # resize image to 28,28
        image = self.features[index].reshape(28,28)

        # change dtype to np.unit8
        image = image.astype(np.uint8)

        # change b&w to colored 

        image = np.stack([image]*3, axis=-1) 
        #  convert array to PIL image
        image = Image.fromarray(image)

        #  Apply tranforms
        image = self.transforms(image)

        # return 
        return image, torch.tensor(self.labels[index],dtype=torch.long)

In [31]:
train_dataset = CustomDataset(X_train,y_train,transforms=custom_transform)
test_dataset = CustomDataset(X_test,y_test,transforms=custom_transform)


In [32]:
train_loader = DataLoader(train_dataset,batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset,batch_size=32, shuffle=True, pin_memory=True)

In [19]:
# fetch pretrained model from torch 
import torchvision.models as models
vgg16 = models.vgg16(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:07<00:00, 74.5MB/s] 


In [22]:
vgg16.classifier

Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=4096, out_features=4096, bias=True)
  (4): ReLU(inplace=True)
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=4096, out_features=1000, bias=True)
)

In [23]:
for param in vgg16.features.parameters():
    param.requires_grad=False


In [24]:
vgg16.classifier = nn.Sequential(
    nn.Linear(25088, 1024),
    nn.ReLU(),
    nn.Dropout(p=0.5),

    nn.Linear(1024, 512),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(512,10)
)

In [25]:
vgg16

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [27]:
vgg16 = vgg16.to(device)

In [28]:
learning_rate = 0.0001
epochs = 10

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(vgg16.classifier.parameters(), lr = learning_rate)


In [35]:
# training loop 

for epoch in range(epochs):
    vgg16.train()
    total_epoch_loss = 0.0
    for batch_features, batch_labels in train_loader:

        batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)
       
        optimizer.zero_grad()

        outputs = vgg16(batch_features)
        loss = criterion(outputs,batch_labels)


        loss.backward()
        optimizer.step()

        total_epoch_loss += loss.item()

        avg_loss = total_epoch_loss/len(train_loader)
    print(f'Epoch: {epoch+1}, loss: {avg_loss:.4f}')



Epoch: 1, loss: 0.3258
Epoch: 2, loss: 0.2171
Epoch: 3, loss: 0.1655
Epoch: 4, loss: 0.1308
Epoch: 5, loss: 0.1054
Epoch: 6, loss: 0.0832
Epoch: 7, loss: 0.0651
Epoch: 8, loss: 0.0566
Epoch: 9, loss: 0.0464
Epoch: 10, loss: 0.0422
